<a href="https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilhamilha-creator/flyrank-ml-assignments/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Refresh / Content Opportunity Scoring is a binary classification task. The objective is to analyze content performance signals to classify each page into one of two discrete categories: 0 for stable/growing or 1 for declining/needs refresh. This supports the ranked action queue by identifying which pages most urgently need editorial attention.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
task_type = "Binary Classification"
classes = {0: "Stable/Growing", 1: "Declining/Needs Refresh"}

print(f"Project Framework Selected: {task_type}")
print(f"Target Output Definition: {classes}")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The model predicts a proxy label derived from observed performance trends. The target is `trend_direction == "down"`, which indicates whether a page's search impressions have declined in the recent 90-day period compared to historical performance. This is a measured proxy for "needs refresh" - pages showing decline patterns are candidates for content review and updates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

# Load the actual starter dataset to show the real target distribution
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Create the actual target label
df['is_declining'] = (df["trend_direction"].str.lower() == "down").astype(int)

print("Sample representation of target classification vector (y):")
print(df['is_declining'].head(10).values)
print(f"\nTarget distribution: {df['is_declining'].value_counts().to_dict()}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Our primary success metric is Precision@50. Since content teams have limited editorial capacity, we need to ensure that the top 50 pages flagged for refresh are actually declining. Precision@50 measures: of the top 50 pages the model prioritizes, what fraction are truly declining? A "good" score would be significantly above the base rate of 0.542 (random selection). The hand-written baseline achieves 0.960 Precision@50, setting a high bar for model performance.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json
import numpy as np

# Load the actual model results from the starter pipeline
with open('outputs/model_results.json', 'r') as f:
    model_results = json.load(f)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

baseline_p50 = model_results["baseline"]["baseline_precision_at_50"]
rf_p50 = model_results["models"]["random_forest"]["precision_at_50"]

print("Success Metric: Precision@50")
print(f"Baseline (hand-written rule): {baseline_p50:.3f}")
print(f"Random Forest model: {rf_p50:.3f}")
print(f"Improvement factor: {rf_p50/baseline_p50:.1f}x")
print(f"\nTarget: Significantly exceed baseline of {baseline_p50:.3f}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

The unit of analysis is one unique content page. Each row represents a single content item with its aggregated search performance features over the 90-day analysis window. This grain matches the decision: content teams review individual pages, not domains or queries.

In [ ]:
import pandas as pd

# Load the actual starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

print("Dataset loaded successfully.")
print(f"Unit of Analysis: {df.shape[0]:,} unique content pages (rows)")
print(f"Features per page: {df.shape[1]} performance signals (columns)")

# Show the first few columns to represent the unit of analysis
print("\nSample of unit of analysis (first 2 pages):")
print(df.iloc[:, :5].head(2))

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Content performance patterns are too complex for simple if-statements. A fixed rule like "refresh if older than 180 days" fails to distinguish between a stable old page and a declining one. Machine learning can weight multiple signals simultaneously - freshness trends, position changes, CTR patterns, and engagement metrics - to identify nuanced decline patterns that hand-coded rules miss.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Show examples of why fixed rules fail
print("Why fixed rules fail - examples from data:")
print("\n1. Fixed rule: 'refresh if content_age_days > 180'")
old_stable = df[(df['content_age_days'] > 180) & (df['trend_direction'] == 'up')]
old_declining = df[(df['content_age_days'] > 180) & (df['trend_direction'] == 'down')]
print(f"   Old but STABLE pages: {len(old_stable)} (would waste refresh effort)")
print(f"   Old and DECLINING pages: {len(old_declining)} (correctly flagged)")

print("\n2. ML advantage: considers multiple signals together")
print("   ML can distinguish old+stable from old+declining by weighting:")
print("   - CTR trends, position changes, engagement patterns")
print("   - Not just age alone")

print("\nFramework logic: ML captures nuanced patterns that if-statements miss.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.